# visu-predict — Colab launcher

End-to-end training of a `visu-predict` model from Google Colab.

1. Clone a branch of the public repo.
2. Install the package (+ optional GNN extras).
3. (optional) Mount Google Drive and download the shared dataset folder.
4. Train via the CLI or the Python API.

Edit `BRANCH` below to test a specific feature branch (e.g. `tier1-pr01-adaptive-embedding`).

In [ ]:
BRANCH = "main"  # change to e.g. "tier1-pr01-adaptive-embedding" to test a PR
!git clone --branch {BRANCH} --depth 1 https://github.com/almo-intellect/visu-predict.git
%cd visu-predict

In [ ]:
# Install the package. `gnn` extras require torch-geometric which can be slow to install.
!pip install -q -e ".[gnn,holidays]"

## Data

Two options:
- **Drive mount + manual paths**: mount your Drive, point `--data` at a CSV there.
- **Download script**: pulls the maintainer's shared Drive folder via `gdown`.

In [ ]:
# Option A: mount Drive and use Drive paths.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Option B: pull the shared dataset folder once into Drive.
# (Skip this if you already have the files in Drive.)
!pip install -q gdown
!python scripts/download_data.py --dest /content/drive/MyDrive/visu-predict/inputs

## Train

Edit `configs/example.yaml` (or supply your own) and run the CLI. Outputs go to whatever `base_output_dir` you set — pointing it at Drive persists checkpoints across Colab sessions.

In [ ]:
# Quick edit: redirect base_output_dir into Drive.
import yaml, pathlib
cfg_path = pathlib.Path('configs/example.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['base_output_dir'] = '/content/drive/MyDrive/visu-predict/outputs'
cfg_path.write_text(yaml.safe_dump(cfg))
print('base_output_dir =', cfg['base_output_dir'])

In [ ]:
# Train. Replace METR-LA.csv with whichever traffic CSV you want.
!visu-predict train \
  --config configs/example.yaml \
  --data /content/drive/MyDrive/visu-predict/inputs/METR-LA.csv

### Enabling the STAE pipeline (Tier 1)

On a branch that includes the STAE upgrade, edit `configs/example.yaml` to set:

```yaml
model_pipeline: stae
use_discrete_time_embeddings: true
steps_per_day: 288
d_input: 24
d_tod: 24
d_dow: 24
d_adaptive: 264
d_node: 0
```

The five `d_*` values must sum to `hidden_dim`.